# PETadex.v260701 Creation: 3. Cleaning seed MSAs for HMM creation

> **Author:** Thomas Quigley\
> **Date Started:** July 21st, 2026\
> **Date Finished:** August 30th, 2026\
> **Input:** \
> s3://petadex/petadex.v260701/petadex-plasticome.v260701.proteins.fastaa\
> s3://petadex/petadex.v260701/1_pazy_cleaning/petadex-plasticome_structures/*.cif\
> s3://petadex/petadex.v260701/2_nr_search/petadex-nr.v260701.significant-hits.fastaa\
> **Output:** \
> s3://petadex/petadex.v260701/3_msa_cleaning/plasticome.v2.sequence_annotations.csv\
> s3://petadex/petadex.v260701/3_msa_cleaning/plasticome_msas/plasticome.component_*.afa

---

## 1. Objective

The goal of this notebook is to create a clean set of multiple sequence alignments (MSAs) that will be used to seed the PETadex-HMMs. To do this, I will first create alignments of close homologs to the Plasticome sequences (DIAMOND <E-50), then remove any sequences that do not contain the canonical catalytic residues of the component, then remove any accessory domains from the alignment.

### 1.1. Place in the Project

In the previous notebook, I had completed the search for plastic-degrading enzyme homologs within the NCBI nr protein database. Although these are predicted to be homologous to plastic-degrading enzymes, we cannot confidently say they contain the domains necessary to catalyze the degradation of plastic polymers. To make that claim, we can use HMMs seeded from the catalytic domains of the verified plastic-degrading enzymes and classify each nr homolog as either containing a complete catalytic domain hit, a partial catalytic domain hit, or no catalytic domain hit. This will subset our data into high quality homologs, lower quality homologs / fragments of high quality homologs, or junk hits.

The distinction between a lower quality homolog and a fragment of a high quality homolog is that the HMM may only hit half the catalytic domain because:
- A: Only half the catalytic domain is high enough homology to hit the HMM --> therefore lower quality homolog
- B: There is a large insertion between the two halves of the catalytic domain, thus only half the domain is hit --> this is a higher quality homolog, we just need to attempt to solve this issue
- C: The edge of the HMM hit on the catalytic domain is the true edge of the catalytic domain --> the catalytic domain is fragmented

In order to classify these HMM hits, we must be confident that the HMMs are high quality, thus the seed MSAs must be of high quality. This notebook is how we know the input MSAs are high quality.

## 2. Experiment
### 2.0. Install requirements
---
1. MUSCLE 5.3.linux64
2. DIAMOND v2.2.4.184

### 2.0.1. Annotating CATH domains
---
In order to make decisions on what a good alignment looks like, I have gone through each CATH domain and attempted to learn as much as I can about them. I have looked into the mechanism for each domain and the annotated the important residues. 

The results of this literature review are in ["plasticome.v2 - cath_domain_annotations"](https://docs.google.com/spreadsheets/d/1SCwQSuyah-NXTzrVx9uX9w_HwPOn-Yzh4pOv3Cy5s5Q/edit?gid=528254709#gid=528254709). I will use these annotations to infer the catalytic mechanism of each component and build reference sequences for each.

### 2.1. Create MSAs of close Plasticome homologs   
---
An issue that I've come across with creating HMMs from a set of very few sequences (<10), is that the seed MSAs suck. There is not enough sequences to identify what the conserved columns are (Figure below). Since we want to create seed MSAs for the PETadex-HMMs, and we need more sequences to make those MSAs, I am going to create alignments of close Plasticome homologs, and then using the larger alignments, properly clean them. By having a greater sequence diversity entering the alignments, I should be able to pick apart the MSAs easier to clean them (more sequences = greater resolution of consensus columns).

![Component 34 Example MSA from only Plasticome Sequences](../resources/260721_issue118_3_msa_cleaning/bad_alignment.png)

### 2.1.1. Cluster close homologs
---
I want to make the HMMs so they represent the input Plasticome sequences as closely as possible. To do this, I'm going to have the Plasticome homologs be very, very homologous to Plasticome (<1.0E-50). Since this will still produce thousands of sequences per component (with very redundant sequence information), I'm going to cluster the sequences (DIAMOND linclust, local) at percentage identities that is determined by a sequence count gradient:
- 50k+ = 30pid
- 10k+ = 50pid
- 100+ = 70pid
- 1+ = 90pid

By tuning the clustering to the sequence count, this will produce a subset of the Plasticome homologs that is small enough to make non-gappy alignments, but large enough to inform which columns are the consensus columns (aiming for about 100-1000 sequences per alignment).

In [ ]:
# ───────────── #
# SCRIPT SET UP #
# ───────────── #
print("Starting up...")

import concurrent.futures
import os
import subprocess

# ─── INPUT ─── #
NR_HITS_PATH = "../s3/2_nr_search/petadex-nr.v260701.significant-hits.fastaa"
"""
Header format:
>genbank_accessions|e-value|pid|bitscore|query_sequence|component_id
Example:
>WP_305403363.1|7.24e-11|23.5|73.6|PL181|C034
"""

PLASTICOME_NON_REDUNDANT_CENTROIDS_PATH = "../s3/petadex-plasticome.v260701.proteins.fastaa"
"""
Header formmat:
>petadex_id|primary_accession|genbank_accessions|pazy_ids|component_id
Example:
>PL2|CCK74972.1|MBQ0729274.1:MBQ0790945.1|2|C001
"""

# ─── WORKING ─── #
HOMOLOG_FASTAA_DIRECTORY = "./3_msa_cleaning/working/close_homolog_fastaas" # header format remains same to NR_HITS and PLASTICOME
HOMOLOG_CLUSTERING_DIRECTORY = "./3_msa_cleaning/working/close_homolog_clustering" # first column is centroid

# ─── OUTPUT ─── #
UNCLEANED_FASTAA_DIRECTORY = "./3_msa_cleaning/working/uncleaned_fastaas" # header format remains same to NR_HITS and PLASTICOME

# ─── PARAMETERS ─── #
EVALUE_CUTOFF = 1.0E-50
N_THREADS = 480 # matches CPU core count of the host
MEMORY_LIMIT_GB = 700 # leaves headroom below the host's ~754Gi total RAM
N_PARALLEL_JOBS = 8 # components clustered concurrently; split the thread & memory budget across them
CLUSTER_ID_THRESHOLDS = ( # (minimum pre-cluster sequence count, %id to cluster at); checked largest-first, so bigger/more redundant components get collapsed harder
    (50_000, 30),
    (10_000, 50),
    (100, 70),
    (0, 90),
)
THREADS_PER_JOB = max(1, N_THREADS // N_PARALLEL_JOBS)
MEMORY_LIMIT_GB_PER_JOB = max(1, MEMORY_LIMIT_GB // N_PARALLEL_JOBS)

# ───────────────────── #
# SELECT CLOSE HOMOLOGS #
# ───────────────────── #

os.makedirs(HOMOLOG_FASTAA_DIRECTORY, exist_ok = True)
component_handles = {}
component_precluster_counts = {}

def get_component_handle(component_id):
    handle = component_handles.get(component_id)
    if handle is None:
        handle = open(os.path.join(HOMOLOG_FASTAA_DIRECTORY, f"close_homologs.{component_id}.fa"), "w")
        component_handles[component_id] = handle
    return handle

# ─── RETRIEVE SEQUENCES BELOW E-VALUE CUTOFF FROM NR HITS ─── #
n_close_homologs = 0
keep_current = False
current_component_id = None
with open(NR_HITS_PATH) as nr_hits:
    for line in nr_hits:
        if line.startswith(">"):
            accession, e_value, pid, bitscore, query_sequence, current_component_id = line[1:].strip().rsplit("|", 5) # accession may itself contain "|", so split from the right
            keep_current = float(e_value) < EVALUE_CUTOFF
            if keep_current:
                n_close_homologs += 1
                component_precluster_counts[current_component_id] = component_precluster_counts.get(current_component_id, 0) + 1

        if keep_current:
            get_component_handle(current_component_id).write(line)

print(f"{n_close_homologs:,} NR hits below {EVALUE_CUTOFF:.1E} retained")

# ─── ADD PLASTICOME SEQUENCES TO THE LIST OF SEQUENCES ─── #
n_plasticome = 0
with open(PLASTICOME_NON_REDUNDANT_CENTROIDS_PATH) as plasticome:
    for line in plasticome:
        if line.startswith(">"):
            current_component_id = line[1:].strip().rsplit("|", 1)[-1] # component_id is always the last field
            n_plasticome += 1
            component_precluster_counts[current_component_id] = component_precluster_counts.get(current_component_id, 0) + 1

        get_component_handle(current_component_id).write(line)

print(f"{n_plasticome:,} Plasticome sequences added")

# ─── CREATE PER COMPONENT FASTAAS ─── #
for handle in component_handles.values():
    handle.close()

component_fastaa_paths = {
    component_id: os.path.join(HOMOLOG_FASTAA_DIRECTORY, f"close_homologs.{component_id}.fa")
    for component_id in component_handles
}
print(f"{len(component_fastaa_paths)} component fastaas written -> {HOMOLOG_FASTAA_DIRECTORY}/")

# ────────────────────── #
# CLUSTER EACH COMPONENT #
# ────────────────────── #

os.makedirs(HOMOLOG_CLUSTERING_DIRECTORY, exist_ok = True)
os.makedirs(UNCLEANED_FASTAA_DIRECTORY, exist_ok = True)

def cluster_id_for_count(n_sequences):
    for min_count, cluster_id in CLUSTER_ID_THRESHOLDS:
        if n_sequences > min_count:
            return cluster_id
    return CLUSTER_ID_THRESHOLDS[-1][1]

def process_component(component_id, component_fastaa_path):
    cluster_id = cluster_id_for_count(component_precluster_counts.get(component_id, 0))

    # ─── DIAMOND CLUSTER ─── #
    component_clustering_output_path = os.path.join(HOMOLOG_CLUSTERING_DIRECTORY, f"close_homologs.{component_id}.tsv")
    subprocess.run(
        [
            "diamond", "linclust",
            "--db", component_fastaa_path,
            "--out", component_clustering_output_path,
            "--id", str(cluster_id),
            "--ext", "full",
            "--comp-based-stats", "0",
            "--no-parse-seqids",
            "--threads", str(THREADS_PER_JOB),
            "--memory-limit", f"{MEMORY_LIMIT_GB_PER_JOB}G", # diamond requires a unit suffix (G/M/K/T); a bare number errors out
        ],
        stdout = subprocess.DEVNULL,
        stderr = subprocess.DEVNULL,
        check = True,
    )
    
    # ─── SUBSET SEQUENCES TO BE CENTROIDS ONLY ─── #
    centroid_ids = set()
    with open(component_clustering_output_path) as clustering_output:
        for line in clustering_output:
            centroid, _member = line.rstrip("\n").split("\t") # first column is centroid
            centroid_ids.add(centroid)

     
    component_uncleaned_fastaa_path = os.path.join(UNCLEANED_FASTAA_DIRECTORY, f"uncleaned.{component_id}.fa")
    keep_current = False
    with open(component_fastaa_path) as component_fastaa, open(component_uncleaned_fastaa_path, "w") as uncleaned_fastaa:
        for line in component_fastaa:
            if line.startswith(">"):
                keep_current = line[1:].strip() in centroid_ids
            if keep_current:
                uncleaned_fastaa.write(line)

    return len(centroid_ids), cluster_id

# ─── RUN COMPONENTS CONCURRENTLY, LARGEST FIRST FOR BETTER LOAD BALANCING ─── #
ordered_component_ids = sorted(component_fastaa_paths, key = lambda cid: component_precluster_counts.get(cid, 0), reverse = True)

component_postcluster_counts = {}
component_cluster_ids = {}
failed_component_ids = []

with concurrent.futures.ThreadPoolExecutor(max_workers = N_PARALLEL_JOBS) as executor:
    futures = {
        executor.submit(process_component, component_id, component_fastaa_paths[component_id]): component_id
        for component_id in ordered_component_ids
    }
    for future in concurrent.futures.as_completed(futures):
        component_id = futures[future]
        try:
            component_postcluster_counts[component_id], component_cluster_ids[component_id] = future.result()
        except Exception as e:
            failed_component_ids.append(component_id)
            print(f"  ! {component_id} failed: {e}")

print(f"{len(component_postcluster_counts)}/{len(component_fastaa_paths)} components clustered -> {UNCLEANED_FASTAA_DIRECTORY}/")
if failed_component_ids:
    print(f"  Failed components: {', '.join(sorted(failed_component_ids))}")

print("Done.")

# ───────────── #
# SUMMARY TABLE #
# ───────────── #

print()
header = f"{'Component':<12}{'Pre-cluster':>14}{'Cluster %':>12}{'Post-cluster':>14}"
print(header)
print("-" * len(header))
for component_id in sorted(component_precluster_counts):
    pre = component_precluster_counts[component_id]
    cluster_id = component_cluster_ids.get(component_id, cluster_id_for_count(pre))
    post = component_postcluster_counts.get(component_id, 0)
    print(f"{component_id:<12}{pre:>14,}{f'{cluster_id}%':>12}{post:>14,}")


Starting up...
1,606,845 NR hits below 1.0E-50 retained
493 Plasticome sequences added
46 component fastaas written -> ./3_msa_cleaning/working/close_homolog_fastaas/
46/46 components clustered -> ./3_msa_cleaning/working/uncleaned_fastaas/
Done.

Component      Pre-cluster   Cluster %  Post-cluster
----------------------------------------------------
C001                28,348         50%           776
C002                 4,656         70%           549
C003               500,583         30%           728
C004                15,424         50%           208
C005                58,961         30%            94
C006                36,820         50%           251
C007                23,243         50%           562
C008                 4,570         70%            71
C009                 7,278         70%           753
C010                 1,689         70%           520
C011                   936         70%           175
C012                11,640         50%           398
C013      

### 2.1.2. Create per component alignments
---
We now have the pre-processed FASTAAs for each component. I am going to create alignments of each of these using the MUSCLE Super5 algorithm. This algorithm speeds up the process of creating the alignment at the cost of accuracy. Since I'm planning on cleaning the alignment sequences and then re-aligning them, this alignment step does not have to be perfect.

**NOTE:** After looking at the alignment of these sequences, specifically at the catalytic residue columns (as per CATH annotations), I have split the following components into sub-components:
- C001-1: PL121
- C001-2: PL69 
- C001-3: PL214
- C001-4: PL120
- C001-5: PL119
- C006-1: PL88
- C008-1: PL46
- C014-1: PL453, PL164, PL166, PL187
- C014-2: PL114
- C014-3: PL74
- C014-4: PL115

I am going to subset these into sub-component alignments.

In [ ]:
# ───────────── #
# SCRIPT SET UP #
# ───────────── #
print("Starting up...")

import concurrent.futures
import os
import subprocess

# ─── INPUT ─── #
UNCLEANED_FASTAA_DIRECTORY = "./3_msa_cleaning/working/uncleaned_fastaas"
"""
Header format:
>genbank_accessions|e-value|pid|bitscore|query_sequence|component_id
Example:
>WP_305403363.1|7.24e-11|23.5|73.6|PL181|C034
"""

PLASTICOME_NON_REDUNDANT_CENTROIDS_PATH = "../s3/petadex-plasticome.v260701.proteins.fastaa"
"""
Header formmat:
>petadex_id|primary_accession|genbank_accessions|pazy_ids|component_id
Example:
>PL2|CCK74972.1|MBQ0729274.1:MBQ0790945.1|2|C001
"""

# ─── WORKING ─── #
SUBCOMPONENT_FASTAA_DIRECTORY = "./3_msa_cleaning/working/subcomponent_fastaas"

# ─── OUTPUT ─── #
UNCLEANED_ALIGNMENTS_DIRECTORY = "./3_msa_cleaning/working/uncleaned_alignments"

# ─── PARAMETERS ─── #
N_THREADS = 480 # matches CPU core count of the host
N_PARALLEL_JOBS = 8 # components aligned concurrently; split the thread budget across them
THREADS_PER_JOB = max(1, N_THREADS // N_PARALLEL_JOBS)

# Per the NOTE above: split these components by Plasticome query sequence hit it (or it is)
SUB_COMPONENT_GROUPS = {
    "C001": {
        "C001-1": {"PL121"},
        "C001-2": {"PL69"},
        "C001-3": {"PL214"},
        "C001-4": {"PL120"},
        "C001-5": {"PL119"},
    },
    "C006": {
        "C006-1": {"PL88"},
    },
    "C008": {
        "C008-1": {"PL46"},
    },
    "C014": {
        "C014-1": {"PL453", "PL164", "PL166", "PL187"},
        "C014-2": {"PL114"},
        "C014-3": {"PL74"},
        "C014-4": {"PL115"},
    },
}

# ────────────────────────────────────────────── #
# RE-ADD PLASTICOME SEQUENCES LOST TO CLUSTERING #
# ────────────────────────────────────────────── #

plasticome_records = {} # petadex_id -> raw fasta lines (header + sequence)
plasticome_component_of = {} # petadex_id -> component_id
current_pl_id = None
with open(PLASTICOME_NON_REDUNDANT_CENTROIDS_PATH) as plasticome:
    for line in plasticome:
        if line.startswith(">"):
            fields = line[1:].strip().split("|") # genbank_accessions can itself contain "|" (e.g. PIR-style ids: "pir||A44761|"), so only the first/last fields are trustworthy
            current_pl_id, component_id = fields[0], fields[-1]
            plasticome_records[current_pl_id] = []
            plasticome_component_of[current_pl_id] = component_id
        plasticome_records[current_pl_id].append(line)

plasticome_ids_by_component = {}
for pl_id, component_id in plasticome_component_of.items():
    plasticome_ids_by_component.setdefault(component_id, set()).add(pl_id)

def plasticome_ids_present(component_fastaa_path):
    present_ids = set()
    with open(component_fastaa_path) as fastaa:
        for line in fastaa:
            if line.startswith(">"):
                pl_id = line[1:].strip().split("|", 1)[0] # NR-hit headers start with a genbank accession, which never collides with a "PLxxx" petadex_id
                if pl_id in plasticome_records:
                    present_ids.add(pl_id)
    return present_ids

def readd_missing_plasticome(component_id, component_fastaa_path):
    expected_ids = plasticome_ids_by_component.get(component_id, set())
    if not expected_ids:
        return 0

    missing_ids = expected_ids - plasticome_ids_present(component_fastaa_path)
    if not missing_ids:
        return 0

    with open(component_fastaa_path, "a") as fastaa:
        for pl_id in missing_ids:
            fastaa.writelines(plasticome_records[pl_id])
    return len(missing_ids)

# ────────────────────────── #
# SUBSET COMPONENTS BY PL ID #
# ────────────────────────── #

os.makedirs(SUBCOMPONENT_FASTAA_DIRECTORY, exist_ok = True)

def pl_id_of(header_line):
    fields = header_line[1:].strip().split("|")
    if len(fields) == 5: # Plasticome header: >petadex_id|primary_accession|genbank_accessions|pazy_ids|component_id
        return fields[0]
    _accession, _e_value, _pid, _bitscore, query_sequence, _component_id = header_line[1:].strip().rsplit("|", 5) # NR-hit header; accession may itself contain "|"
    return query_sequence

def subset_component(component_id, component_fastaa_path):
    groups = SUB_COMPONENT_GROUPS.get(component_id)
    if groups is None:
        return {component_id: component_fastaa_path}

    pl_to_subcomponent = {pl_id: sub_id for sub_id, pl_ids in groups.items() for pl_id in pl_ids}

    handles = {}
    def get_handle(sub_id):
        handle = handles.get(sub_id)
        if handle is None:
            handle = open(os.path.join(SUBCOMPONENT_FASTAA_DIRECTORY, f"uncleaned.{sub_id}.fa"), "w")
            handles[sub_id] = handle
        return handle

    current_sub_id = component_id
    with open(component_fastaa_path) as fastaa:
        for line in fastaa:
            if line.startswith(">"):
                current_sub_id = pl_to_subcomponent.get(pl_id_of(line), component_id) # unmatched -> remainder under original component id
            get_handle(current_sub_id).write(line)

    for handle in handles.values():
        handle.close()

    return {sub_id: os.path.join(SUBCOMPONENT_FASTAA_DIRECTORY, f"uncleaned.{sub_id}.fa") for sub_id in handles}

component_fastaa_paths = {
    filename[len("uncleaned."):-len(".fa")]: os.path.join(UNCLEANED_FASTAA_DIRECTORY, filename)
    for filename in os.listdir(UNCLEANED_FASTAA_DIRECTORY)
    if filename.endswith(".fa")
}
print(f"{len(component_fastaa_paths)} component fastaas found -> {UNCLEANED_FASTAA_DIRECTORY}/")

n_readded = 0
for component_id, component_fastaa_path in component_fastaa_paths.items():
    n_readded += readd_missing_plasticome(component_id, component_fastaa_path)
print(f"{n_readded:,} Plasticome sequences re-added after being dropped by clustering")

subcomponent_fastaa_paths = {}
for component_id, component_fastaa_path in component_fastaa_paths.items():
    subcomponent_fastaa_paths.update(subset_component(component_id, component_fastaa_path))

n_split = sum(1 for component_id in SUB_COMPONENT_GROUPS if component_id in component_fastaa_paths)
print(f"{len(subcomponent_fastaa_paths)} sub-component fastaas ready ({n_split} components split) -> {SUBCOMPONENT_FASTAA_DIRECTORY}/")

# ───────────────────────────────────────────────────────── #
# CREATE A FASTAA FOR ANY SUB-COMPONENT THAT STILL HAS NONE #
# ───────────────────────────────────────────────────────── #
# this would be because there is no homologs in NCBI nr

all_subcomponent_ids = {sub_id for groups in SUB_COMPONENT_GROUPS.values() for sub_id in groups}
missing_subcomponent_ids = sorted(all_subcomponent_ids - set(subcomponent_fastaa_paths))
for sub_id in missing_subcomponent_ids:
    pl_ids = next(groups[sub_id] for groups in SUB_COMPONENT_GROUPS.values() if sub_id in groups)
    sub_fastaa_path = os.path.join(SUBCOMPONENT_FASTAA_DIRECTORY, f"uncleaned.{sub_id}.fa")
    with open(sub_fastaa_path, "w") as fastaa:
        for pl_id in pl_ids:
            fastaa.writelines(plasticome_records[pl_id])
    subcomponent_fastaa_paths[sub_id] = sub_fastaa_path
if missing_subcomponent_ids:
    print(f"{len(missing_subcomponent_ids)} sub-component fastaa(s) created directly from Plasticome sequences (no homologs at all): {', '.join(missing_subcomponent_ids)}")

# ──────────────────────── #
# ALIGN EACH SUB-COMPONENT #
# ──────────────────────── #

os.makedirs(UNCLEANED_ALIGNMENTS_DIRECTORY, exist_ok = True)

def align_component(component_id, component_fastaa_path):
    # ─── MUSCLE ALIGN THE CENTROIDS ─── #
    component_alignment_path = os.path.join(UNCLEANED_ALIGNMENTS_DIRECTORY, f"uncleaned.{component_id}.afa")
    subprocess.run(
        [
            "muscle", "-super5", component_fastaa_path,
            "-output", component_alignment_path,
            "-threads", str(THREADS_PER_JOB),
        ],
        stdout = subprocess.DEVNULL,
        stderr = subprocess.DEVNULL,
        check = True,
    )
    print(f"Finished aligning: {component_id}", flush = True) # threads' stdout is block-buffered under Jupyter, so force a flush or this batches up instead of logging live

    return component_alignment_path

# ─── RUN SUB-COMPONENTS CONCURRENTLY, LARGEST FASTAA FIRST FOR BETTER LOAD BALANCING ─── #
ordered_component_ids = sorted(subcomponent_fastaa_paths, key = lambda cid: os.path.getsize(subcomponent_fastaa_paths[cid]), reverse = True)

n_aligned = 0
failed_component_ids = []

with concurrent.futures.ThreadPoolExecutor(max_workers = N_PARALLEL_JOBS) as executor:
    futures = {
        executor.submit(align_component, component_id, subcomponent_fastaa_paths[component_id]): component_id
        for component_id in ordered_component_ids
    }
    for future in concurrent.futures.as_completed(futures):
        component_id = futures[future]
        try:
            future.result()
            n_aligned += 1
        except Exception as e:
            failed_component_ids.append(component_id)
            print(f"  ! {component_id} failed: {e}")

print(f"{n_aligned}/{len(subcomponent_fastaa_paths)} sub-components aligned -> {UNCLEANED_ALIGNMENTS_DIRECTORY}/")
if failed_component_ids:
    print(f"  Failed components: {', '.join(sorted(failed_component_ids))}")

print("Done.")


Starting up...
46 component fastaas found -> ./3_msa_cleaning/working/uncleaned_fastaas/
0 Plasticome sequences re-added after being dropped by clustering
57 sub-component fastaas ready (4 components split) -> ./3_msa_cleaning/working/subcomponent_fastaas/
Finished aligning: C037
Finished aligning: C001
Finished aligning: C010
Finished aligning: C009
Finished aligning: C032
Finished aligning: C020
Finished aligning: C025
Finished aligning: C019
Finished aligning: C017
Finished aligning: C015
Finished aligning: C029
Finished aligning: C002
Finished aligning: C012
Finished aligning: C036
Finished aligning: C028
Finished aligning: C018
Finished aligning: C011
Finished aligning: C041
Finished aligning: C007
Finished aligning: C006
Finished aligning: C021
Finished aligning: C004
Finished aligning: C046
Finished aligning: C005
Finished aligning: C014-1
Finished aligning: C035
Finished aligning: C043
Finished aligning: C040
Finished aligning: C042
Finished aligning: C024
Finished aligning: C0

### 2.1.3. Create a reference sequence for each sub-component
---
I want to ensure that all the sequences that are entering the seed alignment have the canonical catalytic residues of the component. To do this, I have annotated the catalytic residues for a representative for each CATH domain (["plasticome.v2 - cath_domain_annotations"](https://docs.google.com/spreadsheets/d/1SCwQSuyah-NXTzrVx9uX9w_HwPOn-Yzh4pOv3Cy5s5Q/edit?gid=528254709#gid=528254709)), and then propagated the catalytic residue annotations to a representative of each sub-component through visual inspection (PyMOL). The final annotations are held in ["plasticome.v2 - plasticome.v2.sequence_annotations"](https://docs.google.com/spreadsheets/d/1SCwQSuyah-NXTzrVx9uX9w_HwPOn-Yzh4pOv3Cy5s5Q/edit?gid=814497702#gid=814497702).

In [ ]:
REFERENCE_ANNOTAITONS="../s3/3_msa_cleaning/plasticome.v2.sequence_annotations.csv"

### 2.1.4. Remove sequences that do not align on the catalytic columns
---
I am going to ensure that every single sequence entering the alignment contains the catalytic residues of the CATH domain, to ensure there is not too much functional diversity within the HMMs. This will make it so that the PETadex-HMM seed alignments are limited to only sequences that align properly to the HMM and contain the canonical catalytic residues. This should hopefully improve the quality of the HMMs and constrain the sequence space to be close homologs with functional similarities.

In [4]:
# ───────────── #
# SCRIPT SET UP #
# ───────────── #
print("Starting up...")

import os
import re
import csv

# ─── INPUT ─── #
UNCLEANED_ALIGNMENTS_DIRECTORY = "./3_msa_cleaning/working/uncleaned_alignments"
"""
Header format 1:
>genbank_accessions|e-value|pid|bitscore|query_sequence|component_id
Example:
>WP_305403363.1|7.24e-11|23.5|73.6|PL181|C034

Header format 2:
>petadex_id|primary_accession|genbank_accessions|pazy_ids|component_id
Example:
>PL2|CCK74972.1|MBQ0729274.1:MBQ0790945.1|2|C001
"""

REFERENCE_ANNOTATIONS = "../s3/3_msa_cleaning/plasticome.v2.sequence_annotations.csv"
"""
Table format:
PETadex Domain Name,component_ids,Reference PDB/AF/GenBank,Catalytic Residues,Sequence
alpha/beta hydrolase,C001-1,PL121,"S139,H201,D228",MQVIQQTV
component_ids is comma-separated when one row's reference/annotations apply
to multiple components (e.g. "C001, C033").
"""

# ─── OUTPUT ─── #
CATALYTICALLY_CLEANED_ALIGNMENTS_DIRECTORY = "./3_msa_cleaning/working/catalytically_cleaned_alignments"


# ──────────────────────────── #
# ALIGN TO REFERENCE SEQUENCES #
# ──────────────────────────── #

CATALYTIC_RESIDUE_RE = re.compile(r"^([A-Za-z])(\d+)$")

def read_reference_annotations(annotations_path):
    """
    Reads REFERENCE_ANNOTATIONS and returns dict: component_id -> (accession, catalytic_sites),
    catalytic_sites being a list of (amino_acid, seq_pos) tuples, seq_pos 1-based into that
    row's own Sequence column. A row's component_ids may list more than one component id
    (comma-separated); each gets its own entry pointing at the same reference/sites.

    Each catalytic residue token is validated against the row's Sequence column (e.g. "S139"
    requires Sequence[138] to actually be an S), raising ValueError on the first mismatch.
    """
    annotations = {}
    with open(annotations_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            accession = row["Reference PDB/AF/GenBank"].strip()
            sequence = row["Sequence"].strip()

            catalytic_sites = []
            for token in row["Catalytic Residues"].split(","):
                token = token.strip()
                if not token:
                    continue
                match = CATALYTIC_RESIDUE_RE.match(token)
                if not match:
                    raise ValueError(f"unparseable catalytic residue token '{token}' for accession '{accession}'")
                aa, pos = match.group(1).upper(), int(match.group(2))
                if pos < 1 or pos > len(sequence):
                    raise ValueError(f"catalytic residue position {pos} ('{token}') out of range for accession '{accession}''s {len(sequence)}-residue Sequence")
                found_aa = sequence[pos - 1].upper()
                if found_aa != aa:
                    raise ValueError(f"catalytic residue '{token}' expects '{aa}' at position {pos}, but accession '{accession}' Sequence has '{found_aa}' there")
                catalytic_sites.append((aa, pos))

            for component_id in row["component_ids"].split(","):
                component_id = component_id.strip()
                if component_id:
                    annotations[component_id] = (accession, catalytic_sites)

    return annotations

def parse_fasta_alignment(alignment_path):
    """
    Parses a plain multi-line FASTA alignment (no spaces in headers) into a
    dict: full header (without the leading '>') -> concatenated aligned sequence.
    """
    sequences = {}
    current_header = None
    with open(alignment_path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                current_header = line[1:]
                sequences[current_header] = ""
            else:
                sequences[current_header] += line.strip()
    return sequences

def select_reference_sequence(sequences, accession):
    """
    Finds the aligned sequence belonging to accession (a bare petadex_id, e.g. "PL121")
    among every sequence in the alignment. The accession is matched exactly against the
    first '|'-delimited field of each header. Returns (None, None) if not found.
    """
    for header, aligned_seq in sequences.items():
        if header.split("|", 1)[0] == accession:
            return header, aligned_seq
    return None, None

def seq_pos_to_aln_col(aligned_seq, seq_pos):
    """
    Converts a 1-based residue position in the ungapped sequence to its 1-based
    column number in the alignment, and returns the residue found there.

    This alignment has no insertion states (plain uppercase MUSCLE output, only
    '-' gaps), so every character -- residue or gap -- advances the column;
    only non-gap characters advance the residue count.
    """
    residue_count = 0
    for col, char in enumerate(aligned_seq, start = 1):
        if char != "-":
            residue_count += 1
            if residue_count == seq_pos:
                return col, char
    raise ValueError(f"Sequence position {seq_pos} exceeds the number of residues ({residue_count}) in the aligned sequence.")

os.makedirs(CATALYTICALLY_CLEANED_ALIGNMENTS_DIRECTORY, exist_ok = True)

print(f"Reading reference annotations from {REFERENCE_ANNOTATIONS}...")
reference_annotations = read_reference_annotations(REFERENCE_ANNOTATIONS)
print(f"{len(reference_annotations)} component(s) with reference annotations")

alignment_paths = {
    filename[len("uncleaned."):-len(".afa")]: os.path.join(UNCLEANED_ALIGNMENTS_DIRECTORY, filename)
    for filename in os.listdir(UNCLEANED_ALIGNMENTS_DIRECTORY)
    if filename.endswith(".afa")
}
print(f"{len(alignment_paths)} alignment(s) found -> {UNCLEANED_ALIGNMENTS_DIRECTORY}/")

n_cleaned = 0
skipped_component_ids = []

for component_id, alignment_path in sorted(alignment_paths.items()):
    if component_id not in reference_annotations:
        print(f"[{component_id}] SKIPPED: no reference annotation found")
        skipped_component_ids.append(component_id)
        continue

    accession, catalytic_sites = reference_annotations[component_id]
    sequences = parse_fasta_alignment(alignment_path)

    # ─── CHECK IF REFERENCE SEQUENCE EXISTS IN ALIGNMENT ─── #
    ref_header, ref_aligned_seq = select_reference_sequence(sequences, accession)
    if ref_header is None:
        print(f"[{component_id}] SKIPPED: reference '{accession}' not found among {len(sequences)} aligned sequence(s)")
        skipped_component_ids.append(component_id)
        continue

    # ─── CHECK IF CATLYTIC COLUMNS MATCH FOR REFERENCE ─── #
    catalytic_columns = []  # list[(aa, aln_col)]
    reference_mismatch = False
    for aa, seq_pos in catalytic_sites:
        aln_col, found_aa = seq_pos_to_aln_col(ref_aligned_seq, seq_pos)
        if found_aa.upper() != aa:
            print(f"[{component_id}] ERROR: reference '{accession}' has '{found_aa}' at seq pos {seq_pos} (column {aln_col}), expected '{aa}'")
            reference_mismatch = True
        catalytic_columns.append((aa, aln_col))
    if reference_mismatch:
        skipped_component_ids.append(component_id)
        continue

    # ─── PRINT THE IDENTITY BREAKDOWN OF CATALYTIC COLUMNS ─── #
    n_sequences = len(sequences)
    print(f"[{component_id}] reference '{accession}' -> {', '.join(f'{aa}{col}' for aa, col in catalytic_columns)} ({n_sequences} sequences)")
    for aa, aln_col in catalytic_columns:
        n_matching = sum(1 for seq in sequences.values() if seq[aln_col - 1].upper() == aa)
        print(f"[{component_id}]   column {aln_col} ({aa}): {n_matching}/{n_sequences} match")

    # ─── REMOVE ANY SEQUENCE THAT DOES NOT ALIGN ON CATALYTIC RESIDUE ─── #
    kept_sequences = {
        header: seq
        for header, seq in sequences.items()
        if all(seq[aln_col - 1].upper() == aa for aa, aln_col in catalytic_columns)
    }

    cleaned_path = os.path.join(CATALYTICALLY_CLEANED_ALIGNMENTS_DIRECTORY, f"catalytically_cleaned.{component_id}.afa")
    with open(cleaned_path, "w") as out:
        for header, seq in kept_sequences.items():
            out.write(f">{header}\n{seq}\n")

    n_removed = n_sequences - len(kept_sequences)
    print(f"[{component_id}] {len(kept_sequences)}/{n_sequences} sequences kept ({n_removed} removed) -> {cleaned_path}")
    n_cleaned += 1

print(f"\n{n_cleaned}/{len(alignment_paths)} components cleaned -> {CATALYTICALLY_CLEANED_ALIGNMENTS_DIRECTORY}/")
if skipped_component_ids:
    print(f"  Skipped components: {', '.join(sorted(skipped_component_ids))}")

print("Done.")


Starting up...
Reading reference annotations from ../s3/3_msa_cleaning/plasticome.v2.sequence_annotations.csv...
57 component(s) with reference annotations
57 alignment(s) found -> ./3_msa_cleaning/working/uncleaned_alignments/
[C001] reference 'PL1' -> S6862, D7991, H8296 (885 sequences)
[C001]   column 6862 (S): 854/885 match
[C001]   column 7991 (D): 816/885 match
[C001]   column 8296 (H): 824/885 match
[C001] 794/885 sequences kept (91 removed) -> ./3_msa_cleaning/working/catalytically_cleaned_alignments/catalytically_cleaned.C001.afa
[C001-1] reference 'PL121' -> S442, H510, D537 (30 sequences)
[C001-1]   column 442 (S): 30/30 match
[C001-1]   column 510 (H): 11/30 match
[C001-1]   column 537 (D): 30/30 match
[C001-1] 11/30 sequences kept (19 removed) -> ./3_msa_cleaning/working/catalytically_cleaned_alignments/catalytically_cleaned.C001-1.afa
[C001-2] reference 'PL69' -> S247, D384, H413 (15 sequences)
[C001-2]   column 247 (S): 14/15 match
[C001-2]   column 384 (D): 13/15 match


### 2.1.5. Manually clean the alignments
---
Now that we have small-ish alignments for each component, I can go through them and start to clean them up. The major thing I cleaned in these alignments is removing accessory domains. These accessory domains are why each alignment took soooo long to run (overnight), as it forced the alignments to be upwards of 50 000 columns long to fit the unalignable accessory domains. These domains are why I was forced to use the less accurate Super5 algorithm for the alignments.

In order to remove any accessory domains, I went to the first column that was highly conserved for a residue (no distinct cut-off, but generally >50% conservation and >50% of maximum quality), and removed every residue left of it. I then repeated this with the last column that was highly conserved, and removed every residue to the right of this. By having the first and last residue of each alignment be highly conserved for the domain I'm looking for, I'm hoping that it limits envelope creep.

In [ ]:
CLEANED_ALIGNMENTS_DIRECTORY = "./3_msa_cleaning/working/accessory_domain_cleaned_alignments" # file names are: {sub-component}.fa

### 2.2. Create the final alignments
---
I now have alignments of each component and sub-component which have any accessory domain removed from them. This will allow me remake the alignments using the PPP alignment algorithm instead of the Super5 algorithm, improving the quality of the alignments.

In [5]:
# ───────────── #
# SCRIPT SET UP #
# ───────────── #
print("Starting up...")

import concurrent.futures
import os
import subprocess

# ─── INPUT ─── #
CLEANED_ALIGNMENTS_DIRECTORY = "./3_msa_cleaning/working/accessory_domain_cleaned_alignments" # file names are: {sub-component}.fa, gapped ('-') protein alignment, header format unchanged from 2.1.2

# ─── WORKING ─── #
DEGAPPED_FASTAA_DIRECTORY = "./3_msa_cleaning/working/cleaned_fastaas"

# ─── OUTPUT ─── #
PLASTICOME_MSAS = "../s3/3_msa_cleaning/plasticome_msas"

# ─── PARAMETERS ─── #
N_THREADS = 480 # matches CPU core count of the host
N_PARALLEL_JOBS = 8 # components aligned concurrently; split the thread budget across them
THREADS_PER_JOB = max(1, N_THREADS // N_PARALLEL_JOBS)

# ─────────────────────────────── #
# REMOVE '-' AND MAKE NEW FASTAAS #
# ─────────────────────────────── #

os.makedirs(DEGAPPED_FASTAA_DIRECTORY, exist_ok = True)

def degap_fastaa(component_id, alignment_path):
    degapped_path = os.path.join(DEGAPPED_FASTAA_DIRECTORY, f"{component_id}.fa")
    with open(alignment_path) as alignment, open(degapped_path, "w") as degapped:
        for line in alignment:
            if line.startswith(">"):
                degapped.write(line)
            else:
                degapped.write(line.strip().replace("-", "") + "\n")
    return degapped_path

component_alignment_paths = {
    filename[:-len(".fa")]: os.path.join(CLEANED_ALIGNMENTS_DIRECTORY, filename)
    for filename in os.listdir(CLEANED_ALIGNMENTS_DIRECTORY)
    if filename.endswith(".fa")
}
print(f"{len(component_alignment_paths)} cleaned alignments found -> {CLEANED_ALIGNMENTS_DIRECTORY}/")

degapped_fastaa_paths = {
    component_id: degap_fastaa(component_id, alignment_path)
    for component_id, alignment_path in component_alignment_paths.items()
}
print(f"{len(degapped_fastaa_paths)} degapped fastaas written -> {DEGAPPED_FASTAA_DIRECTORY}/")

# ──────────────────────────────── #
# RUN MUSCLE ALIGN (PPP ALGORITHM) #
# ──────────────────────────────── #

os.makedirs(PLASTICOME_MSAS, exist_ok = True)

def align_component(component_id, degapped_fastaa_path):
    component_alignment_path = os.path.join(PLASTICOME_MSAS, f"plasticome.component_{component_id}.afa")
    subprocess.run(
        [
            "muscle", "-align", degapped_fastaa_path,
            "-output", component_alignment_path,
            "-threads", str(THREADS_PER_JOB),
        ],
        stdout = subprocess.DEVNULL,
        stderr = subprocess.DEVNULL,
        check = True,
    )
    print(f"Finished aligning: {component_id}", flush = True) # threads' stdout is block-buffered under Jupyter, so force a flush or this batches up instead of logging live

    return component_alignment_path

# ─── RUN COMPONENTS CONCURRENTLY, LARGEST FASTAA FIRST FOR BETTER LOAD BALANCING ─── #
ordered_component_ids = sorted(degapped_fastaa_paths, key = lambda cid: os.path.getsize(degapped_fastaa_paths[cid]), reverse = True)

n_aligned = 0
failed_component_ids = []

with concurrent.futures.ThreadPoolExecutor(max_workers = N_PARALLEL_JOBS) as executor:
    futures = {
        executor.submit(align_component, component_id, degapped_fastaa_paths[component_id]): component_id
        for component_id in ordered_component_ids
    }
    for future in concurrent.futures.as_completed(futures):
        component_id = futures[future]
        try:
            future.result()
            n_aligned += 1
        except Exception as e:
            failed_component_ids.append(component_id)
            print(f"  ! {component_id} failed: {e}")

print(f"{n_aligned}/{len(degapped_fastaa_paths)} components aligned -> {PLASTICOME_MSAS}/")
if failed_component_ids:
    print(f"  Failed components: {', '.join(sorted(failed_component_ids))}")

print("Done.")


Starting up...
57 cleaned alignments found -> ./3_msa_cleaning/working/accessory_domain_cleaned_alignments/
57 degapped fastaas written -> ./3_msa_cleaning/working/cleaned_fastaas/
Finished aligning: C032
Finished aligning: C010
Finished aligning: C020
Finished aligning: C009
Finished aligning: C037
Finished aligning: C025
Finished aligning: C007
Finished aligning: C019
Finished aligning: C017
Finished aligning: C029
Finished aligning: C015
Finished aligning: C002
Finished aligning: C001
Finished aligning: C041
Finished aligning: C036
Finished aligning: C011
Finished aligning: C018
Finished aligning: C006
Finished aligning: C004
Finished aligning: C014-1
Finished aligning: C021
Finished aligning: C046
Finished aligning: C043
Finished aligning: C022
Finished aligning: C040
Finished aligning: C028
Finished aligning: C042
Finished aligning: C038
Finished aligning: C014
Finished aligning: C024
Finished aligning: C008
Finished aligning: C035
Finished aligning: C030
Finished aligning: C005
F

KeyboardInterrupt: 

C016 was taking much too much time with the PPP algorithm, so I used the Super5 algorithm with it instead.

## 3. Summary of Results

I clustered close Plasticome homologs (DIAMOND v2.2.4.184 <1.0E-50) in order to create the seed alignments. To have enough sequences for the alignments, but not too many that the alignments grew too large, the perecent identity clustering (Muscle v5.3, Super5) was determined by the count of sequences in each component (50k+ = 30pid, 10k+ = 50pid, 100+ = 70pid, 1+ = 90pid). The resulting alignments were separated into sub-components, sequences that did not align to the catalytic residues were removed, and accessory domains were cleaved from the alignments. The final sets of cleaned sequences were then aligned once again (Muscle v5.3, PPP or Super5).

**NOTE FROM 4_HMM_CREATION:** C001-5 and C008-1 have about 1 homolog in NCBI nr each, and I don't think creating an HMM off 1 sequence is worth while at all. On further investigation, it appears that PL119 (C001-5 represnetative), is actually a framgent, which somehow had activity (guessing the 6xHis tag helped?), while PL46 I have no idea what the mechanism is.

Since I cannot be confident in the mechanism, and I don't have enough sequences, I am going to remove these two sub-components.

Additionally, I investigated the two components that suspiciously was missing catalytic residues, C001-4 (top image) and C014-4, and I found that C001-4 is a fragment (top image), and C014-4 I have no idea what that mechanism could be. I'm going to not make HMM for these, as I have no idea how to measure the quality.

![C001-4 fragment](../resources/260721_issue118_3_msa_cleaning/C001-4_fragment.png)

![C001-4 fragment](../resources/260721_issue118_3_msa_cleaning/C014-4_no-mechanism.png)

I have kept these sub-component annotations in the reference sheet.